# Partie 2: TDA

## 1) Construction de features topologiques

### Objectif

Ce notebook a pour but d'extraire des descripteurs topologiques à partir des données préparées pour le stage.

L'objectif général est de transformer une structure de données complexe (nuage de points, matrice de distances, graphe de voisinage, embeddings, etc.) en représentations topologiques exploitables dans un pipeline de machine learning.

### Questions visées

À travers ce notebook, on cherche à répondre aux questions suivantes :

- quelle structure topologique peut-on extraire des données ?
- comment construire un objet adapté à la TDA ?
- quels diagrammes de persistance obtient-on ?
- comment convertir ces diagrammes en variables numériques utilisables pour la prédiction ?
- quelles sont les limites et précautions d'interprétation ?

### Remarque

Ce notebook constitue un cadre de travail adaptable.
Les choix précis de représentation, de filtration, de dimension homologique et de vectorisation devront être ajustés en fonction :
- du type de données ;
- du niveau d'observation (patient, échantillon, cellule, sous-groupe, etc.) ;
- de la question scientifique posée.

## 2) Imports et Configuration

In [ ]:
# Imports standards
from pathlib import Path
import sys

# Imports scientifiques
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Imports TDA
from ripser import ripser
from persim import plot_diagrams

# Imports GUDHI pour la vectorisation
from gudhi.representations import PersistenceImage, Landscape, BettiCurve

# Configuration d'affichage
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 120)

# Ajout de la racine du projet au PYTHONPATH si nécessaire
PROJECT_ROOT = Path.cwd().resolve().parents[0]
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

# Imports depuis le projet
from src.data import get_project_root, get_data_dirs, ensure_directories

# Vérification rapide
print("Racine du projet :", get_project_root())
print("Répertoires de données :", get_data_dirs())

ensure_directories()

## 3) Chargement des données de travail/Code

On charge ici les données qui serviront à construire les objets topologiques.

Selon le cas, cela peut être :
- une matrice d'observations ;
- un embedding déjà calculé ;
- une matrice de distances ;
- des coordonnées réduites ;
- un sous-ensemble d'échantillons ;
- ou des données déjà prétraitées dans un notebook précédent.

L'important est de savoir clairement :
- quelle est l'unité d'analyse ;
- quelles colonnes sont utilisées ;
- si les données sont déjà normalisées ou non.

In [ ]:
# Récupération des répertoires standards
DATA_DIRS = get_data_dirs()

# Adapter ici le nom du fichier à ton cas réel
FILENAME = "tda_input.csv"

# On suppose ici que le fichier est placé dans data/processed
DATA_PATH = DATA_DIRS["processed"] / FILENAME

print("Chemin du fichier :", DATA_PATH)
print("Le fichier existe :", DATA_PATH.exists())

# Chargement
df = pd.read_csv(DATA_PATH)

# Aperçu
print("Dimensions :", df.shape)
display(df.head())

## 4) Définition de la vue topologique

Avant de calculer des invariants topologiques, il faut définir l'objet mathématique étudié.

Plusieurs choix sont possibles :
- utiliser directement les variables numériques comme nuage de points ;
- construire une matrice de distances ;
- travailler dans un espace latent ou réduit ;
- construire un graphe de voisinage puis en dériver une structure filtrée.

Dans cette première version, on adopte une approche simple :
on sélectionne un sous-ensemble de variables numériques pour former un nuage de points.

## Sélection des variables

In [ ]:
# Sélection des variables numériques uniquement
colonnes_numeriques = df.select_dtypes(include=[np.number]).columns.tolist()

print("Nombre de colonnes numériques :", len(colonnes_numeriques))
print("Colonnes numériques sélectionnées :")
for col in colonnes_numeriques:
    print("-", col)

# Adaptation éventuelle : choisir un sous-ensemble de variables
FEATURE_COLUMNS = colonnes_numeriques.copy()

# Exemple si tu veux sélectionner à la main :
# FEATURE_COLUMNS = ["var1", "var2", "var3"]

# Construction de la matrice de travail
X = df[FEATURE_COLUMNS].dropna().to_numpy()

print("Shape de X :", X.shape)

## 5) Prétraitement avant TDA

Les calculs topologiques sont sensibles à l'échelle des variables.
Il est donc souvent nécessaire de :
- centrer / réduire ;
- filtrer les colonnes inutiles ;
- supprimer les lignes trop incomplètes ;
- éventuellement limiter la taille du jeu de données pour un premier test.

Dans ce notebook, on commence par une standardisation simple.

In [ ]:
##=========== STANDARDISATION SIMPLE ==============#

from sklearn.preprocessing import StandardScaler

# Standardisation des variables
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("Shape après standardisation :", X_scaled.shape)

# Vérification rapide
print("Moyenne approx. de la première variable :", X_scaled[:, 0].mean())
print("Écart-type approx. de la première variable :", X_scaled[:, 0].std())

##=====================================================#

In [ ]:
##============= Sous-échantillonnage optionnel =========#
#Le calcul de persistance peut devenir coûteux si le nuage de points est trop grand.
#Pour les premiers essais, il peut être utile de travailler sur un sous-échantillon.
# Paramètre optionnel de sous-échantillonnage
USE_SUBSAMPLE = True
MAX_POINTS = 300

if USE_SUBSAMPLE and X_scaled.shape[0] > MAX_POINTS:
    rng = np.random.default_rng(42)
    indices = rng.choice(X_scaled.shape[0], size=MAX_POINTS, replace=False)
    X_tda = X_scaled[indices]
    print(f"Sous-échantillon utilisé : {X_tda.shape[0]} points")
else:
    X_tda = X_scaled
    print(f"Jeu complet utilisé : {X_tda.shape[0]} points")

##========================================================#

## 6) Calcul de Persistance 
### Calcul des diagrammes de persistance

On calcule ici les diagrammes de persistance à partir du nuage de points.

Dans cette première version :
- on utilise `ripser` ;
- on travaille sur une filtration de Vietoris-Rips ;
- on calcule les diagrammes jusqu'à une dimension homologique fixée.

Rappel interprétatif :
- $H_0$ : composantes connexes ;
- $H_1$ : cycles / trous 1D ;
- $H_2$ : cavités, si calculées.

In [ ]:
##====================== Calcul avec Ripser ======================#

# Paramètre de dimension homologique maximale
MAXDIM = 1

# Calcul des diagrammes de persistance
tda_result = ripser(X_tda, maxdim=MAXDIM)

# Les diagrammes sont stockés ici
diagrams = tda_result["dgms"]

print("Nombre de diagrammes calculés :", len(diagrams))

for dim, dgm in enumerate(diagrams):
    print(f"Dimension H{dim} : {dgm.shape[0]} points dans le diagramme")

##====================================================================#

## 7) Visualisation des diagrammes/Code
### Visualisation des diagrammes de persistance

Cette étape permet une première lecture qualitative :
- persistance des composantes ;
- présence ou non de structures cycliques ;
- niveau de bruit topologique apparent.

In [ ]:
plt.figure(figsize=(7, 6))
plot_diagrams(diagrams, show=True)
plt.title("Diagrammes de persistance")
plt.show()

## 8) Premiers résumés numériques
Avant même la vectorisation avancée, on peut calculer quelques résumés simples :

- nombre de points par diagramme ;
- persistance maximale ;
- persistance moyenne ;
- somme des persistances.

Ces indicateurs ne remplacent pas une vraie vectorisation, mais ils sont utiles pour une première intuition.

In [ ]:
##====================== Résumés simples des diagrammes ===================#

def summarize_diagram(diagram: np.ndarray, homology_dim: int) -> dict:
    """
    Résume un diagramme de persistance par quelques statistiques simples.
    """
    if diagram.size == 0:
        return {
            "homology_dim": homology_dim,
            "n_points": 0,
            "max_persistence": 0.0,
            "mean_persistence": 0.0,
            "sum_persistence": 0.0,
        }
    
    # Calcul des persistances
    persistence = diagram[:, 1] - diagram[:, 0]
    
    # Gestion des éventuels infinis
    persistence = persistence[np.isfinite(persistence)]
    
    if len(persistence) == 0:
        return {
            "homology_dim": homology_dim,
            "n_points": 0,
            "max_persistence": 0.0,
            "mean_persistence": 0.0,
            "sum_persistence": 0.0,
        }

    return {
        "homology_dim": homology_dim,
        "n_points": len(persistence),
        "max_persistence": float(np.max(persistence)),
        "mean_persistence": float(np.mean(persistence)),
        "sum_persistence": float(np.sum(persistence)),
    }

resume_diagrams = pd.DataFrame(
    [summarize_diagram(dgm, dim) for dim, dgm in enumerate(diagrams)]
)

display(resume_diagrams)

## 9) Préparation à la vectorisation
### Vectorisation des diagrammes

Pour injecter l'information topologique dans un pipeline ML, il faut convertir les diagrammes de persistance en vecteurs numériques de taille fixe.

Plusieurs options existent :
- persistence images ;
- landscapes ;
- Betti curves ;
- résumés statistiques ;
- kernels ou distances entre diagrammes.

Dans cette première version, on teste trois représentations simples :
- persistence image ;
- landscape ;
- Betti curve.

In [ ]:
##======================= Nettoyage des diagrammes pour la Vectorisation ==================

def clean_diagram_for_vectorization(diagram: np.ndarray) -> np.ndarray:
    """
    Nettoie un diagramme avant vectorisation :
    - suppression des lignes non finies ;
    - retour d'un tableau 2D compatible.
    """
    if diagram.size == 0:
        return np.empty((0, 2))
    
    mask = np.isfinite(diagram).all(axis=1)
    cleaned = diagram[mask]
    
    if cleaned.size == 0:
        return np.empty((0, 2))
    
    return cleaned

diagrams_clean = [clean_diagram_for_vectorization(dgm) for dgm in diagrams]

for dim, dgm in enumerate(diagrams_clean):
    print(f"H{dim} nettoyé :", dgm.shape)

In [ ]:
##================= Vectorisation 1 : Persistence Image ===========#

# La persistence image transforme un diagramme en une représentation vectorielle de taille fixe.
# C'est souvent une bonne première option pour injecter de l'information topologique dans un modèle supervisé.

# On travaille ici sur H1 si disponible, sinon H0
DIM_TO_VECTORIZE = 1 if len(diagrams_clean) > 1 else 0
diagram_target = diagrams_clean[DIM_TO_VECTORIZE]

print("Dimension utilisée pour la vectorisation :", DIM_TO_VECTORIZE)

if diagram_target.shape[0] > 0:
    pi = PersistenceImage(
        bandwidth=0.1,
        resolution=[20, 20]
    )
    
    pi_features = pi.fit_transform([diagram_target])
    pi_vector = pi_features[0]

    print("Shape Persistence Image :", pi_vector.shape)
else:
    pi_vector = None
    print("Diagramme vide : impossible de calculer une persistence image.")

In [ ]:
##================== Vectorisation 2 : Persistence Landscape ===================

#La persistence landscape produit une représentation fonctionnelle discrétisée du diagramme.
#Elle peut être utile pour résumer la géométrie globale de la persistance.

if diagram_target.shape[0] > 0:
    landscape = Landscape(num_landscapes=5, resolution=100)
    landscape_features = landscape.fit_transform([diagram_target])
    landscape_vector = landscape_features[0]

    print("Shape Landscape :", landscape_vector.shape)
else:
    landscape_vector = None
    print("Diagramme vide : impossible de calculer une landscape.")

In [ ]:
##=================== Vectorisation 3 : Betti Curve =============================

#La Betti curve décrit l'évolution du nombre de classes topologiques au cours de la filtration.
#Elle fournit une représentation simple et parfois très interprétable.

if diagram_target.shape[0] > 0:
    betti = BettiCurve(resolution=100)
    betti_features = betti.fit_transform([diagram_target])
    betti_vector = betti_features[0]

    print("Shape Betti Curve :", betti_vector.shape)
else:
    betti_vector = None
    print("Diagramme vide : impossible de calculer une Betti curve.")

## 10) Construction d'une table de features topologiques

Dans un pipeline réel, les features topologiques devront être calculées :
- par individu ;
- par groupe ;
- par état ;
- ou par fenêtre temporelle.

Dans cette première version, on construit simplement une ligne de features pour l'objet topologique analysé ici.

In [ ]:
##=============== Table des features ================

#Dictionnaire de features topologiques
topo_features = {}

# Ajout des résumés simples
for _, row in resume_diagrams.iterrows():
    dim = int(row["homology_dim"])
    topo_features[f"h{dim}_n_points"] = row["n_points"]
    topo_features[f"h{dim}_max_persistence"] = row["max_persistence"]
    topo_features[f"h{dim}_mean_persistence"] = row["mean_persistence"]
    topo_features[f"h{dim}_sum_persistence"] = row["sum_persistence"]

# Ajout éventuel de quelques composantes vectorisées
# Ici on ne garde qu'un extrait pour éviter une table énorme dans ce premier notebook
if pi_vector is not None:
    for i, value in enumerate(pi_vector[:20]):
        topo_features[f"pi_{i}"] = float(value)

if landscape_vector is not None:
    for i, value in enumerate(landscape_vector[:20]):
        topo_features[f"landscape_{i}"] = float(value)

if betti_vector is not None:
    for i, value in enumerate(betti_vector[:20]):
        topo_features[f"betti_{i}"] = float(value)

# Conversion en DataFrame
df_topo_features = pd.DataFrame([topo_features])

print("Shape de la table de features topologiques :", df_topo_features.shape)
display(df_topo_features.head())

## 11) Sauvegarde des features/Code

Les features extraites peuvent ensuite être sauvegardées pour être réutilisées dans :
- un notebook de modélisation ;
- un pipeline supervisé ;
- une phase d'analyse comparative.

In [ ]:
# Nom du fichier de sortie à adapter
OUTPUT_FILENAME = "topological_features.csv"
OUTPUT_PATH = get_data_dirs()["processed"] / OUTPUT_FILENAME

df_topo_features.to_csv(OUTPUT_PATH, index=False)

print("Features sauvegardées dans :", OUTPUT_PATH)

## 12) Limites et précautions

Les résultats de ce notebook doivent être interprétés avec prudence.

Points d'attention :
- la TDA est sensible au choix de la représentation d'entrée ;
- la normalisation influence fortement les résultats ;
- les diagrammes peuvent contenir du bruit topologique ;
- un calcul sur un seul objet ou sur un sous-échantillon ne suffit pas à conclure scientifiquement ;
- la vraie valeur du pipeline viendra de la reproductibilité, de la robustesse et du lien avec la question clinique / biologique.

# Conclusion

Ce notebook permet de poser une première base pour l'extraction de descripteurs topologiques.

Les prochaines étapes possibles sont :
- calculer ces features par individu ou par sous-groupe ;
- comparer plusieurs constructions topologiques ;
- tester différentes vectorisations ;
- intégrer les features topologiques dans un pipeline de prédiction ;
- étudier leur stabilité et leur interprétabilité.